In [ ]:
/**
 * ============================================================================
 * REMOVING SINGLE POINTS OF FAILURE (SPOF) — EXPLAINED LIKE WE'RE TALKING
 * ============================================================================
 *
 * Picture this: your app has been running smoothly for months. Then one
 * random server dies. Just ONE. And suddenly your ENTIRE app is down,
 * thousands of users are stuck, and someone's phone is buzzing at 3 AM.
 *
 * That one server was a "Single Point of Failure" — usually shortened to
 * SPOF (pronounced like "spoff"). It means: one piece breaks, and the
 * whole system (or a really important part of it) breaks with it.
 *
 * The scary part? SPOFs love to hide. Your diagram might show multiple
 * app servers, multiple caches, multiple queues... and still have ONE
 * database, ONE DNS provider, or ONE load balancer quietly holding
 * everything together. Looks redundant on paper, isn't redundant in reality.
 *
 * This file walks through:
 *   1. What a SPOF actually is
 *   2. Why they're so dangerous
 *   3. How to go hunting for them
 *   4. The main strategies to remove them
 *   5. How to remove them layer-by-layer (DNS, app, database, etc.)
 *   6. How to actually TEST that your fixes work
 *   7. How to decide when it's even worth fixing one
 *
 *
 * ============================================================================
 * WHAT IS A SINGLE POINT OF FAILURE?
 * ============================================================================
 *
 * Simple definition: a SPOF is any single piece of your system that, if it
 * breaks, takes down the whole system (or a critical part of it) — with
 * NO backup plan automatically kicking in.
 *
 * Important nuance: the problem isn't "there's only ONE of something."
 * The real problem is "there's only one, AND there's no working backup
 * path, AND nothing takes over automatically, AND there's no acceptable
 * fallback mode." All of those missing together = a SPOF.
 *
 * ARCHITECTURE PICTURE:
 *
 *      [User] ---> [Web Server] ---> [ONE Database]
 *                                          |
 *                              if THIS breaks, everything breaks
 *
 * SNEAKY EXAMPLE: You might have TWO app servers (nice, redundant!) but
 * both of them talk to the SAME single database. So you've only solved
 * half the problem — the database is still your SPOF.
 *
 * COMMON PLACES SPOFS HIDE:
 *
 *   Layer        | Example SPOF           | What breaks if it fails
 *   -------------|-------------------------|---------------------------
 *   Network      | One load balancer       | ALL traffic gets blocked
 *   Application  | One app server          | No requests get processed
 *   Database     | One database instance   | ALL data becomes unreachable
 *   Storage      | One disk/volume         | Data loss or unavailability
 *   DNS          | One DNS provider        | Your domain becomes unreachable
 *   Region       | One data center         | Total outage
 *   People       | One expert who "knows"  | Nobody else can fix the issue
 *
 *
 * ============================================================================
 * WHY SPOFs ARE SO DANGEROUS (4 reasons)
 * ============================================================================
 *
 * 1) FAILURE BECOMES VISIBLE TO USERS
 *    Every piece of hardware and software fails eventually — disks wear
 *    out, servers crash, networks glitch. That's normal and unavoidable.
 *    But if there's a SPOF sitting on the critical path, a small,
 *    everyday failure turns into a big, visible outage for your users.
 *
 * 2) IT NEVER FAILS AT A CONVENIENT TIME
 *    You don't get to schedule when a SPOF breaks. It'll pick Black
 *    Friday. It'll pick your big product launch. It'll pick 3 AM when
 *    your best engineers are asleep. This is basically "Murphy's Law for
 *    distributed systems."
 *
 * 3) FAILURES CASCADE (ONE BREAK CAUSES MORE BREAKS)
 *    A SPOF rarely stays contained. It knocks over the next thing, which
 *    knocks over the next thing:
 *
 *    ARCHITECTURE PICTURE (cascading failure):
 *
 *       [Database] --fails-->  [Order Service]     --fails--> [Checkout]
 *                    \-fails--> [Inventory Service] --fails--> [Product Pages]
 *
 *    One database going down just took out checkout AND product pages.
 *    That's the "blast radius" of a SPOF — it's often way bigger than
 *    the single component that actually broke.
 *
 * 4) RECOVERY IS SLOW
 *    Without a backup, "recovery" means someone has to actually go fix
 *    or replace the broken thing before anything works again. WITH a
 *    backup (proper redundancy), traffic can just shift over to the
 *    healthy component immediately, and the team can calmly fix the
 *    broken one without users even noticing.
 *
 *
 * ============================================================================
 * HOW TO FIND SPOFs (before you can fix them, you have to find them)
 * ============================================================================
 *
 * The tricky part is that SPOFs hide behind assumptions like "the cloud
 * provider handles that" or "that service never goes down." Spoiler:
 * everything goes down eventually.
 *
 * THE "WHAT IF" EXERCISE
 * Go through every single component in your system and ask:
 * "What happens if THIS one breaks?"
 *
 *   Component          | If there's only ONE     | If it's redundant
 *   --------------------|--------------------------|---------------------------
 *   Load Balancer       | ALL traffic stops        | Traffic reroutes to another
 *   App Server          | App goes down            | Other servers pick up load
 *   Database            | Reads AND writes fail    | Replicas keep reads working
 *   Cache               | DB might get overwhelmed | Requests degrade gracefully
 *
 *   Important trick: don't stop at "the cache failed." Ask the FOLLOW-UP
 *   question too: "if the cache fails and ALL traffic suddenly hits the
 *   database directly, can the database survive that?" If not, your
 *   "small" cache failure just became a full database failure too.
 *
 * DEPENDENCY MAPPING
 * Draw your whole architecture, trace the path from "user" to "data," and
 * mark anywhere there's only ONE instance of something.
 *
 * ARCHITECTURE PICTURE:
 *
 *   [Users] -> [DNS] -> [Load Balancer] -> [Server 1] -\
 *                                        -> [Server 2] --> [Cache] -> [Primary DB]
 *
 *   Even though there are 2 servers here (nice!), DNS, Load Balancer,
 *   Cache, and the Database are all SINGLE — meaning they're likely
 *   SPOFs unless you've specifically made THEM redundant too.
 *
 *   LESSON: redundancy at one layer does NOT automatically mean
 *   redundancy at every layer. You have to check each one separately.
 *
 * QUESTIONS TO ASK FOR EACH COMPONENT:
 *   - Is there only ONE of this on the critical path?
 *   - If it dies, does something take over AUTOMATICALLY, or does a
 *     human have to notice and fix it manually? (manual = slower)
 *   - How long would recovery actually take — minutes, or hours?
 *   - What's the "blast radius"? (a broken login service might break
 *     EVERYTHING, while a broken "recommended for you" widget barely
 *     matters)
 *   - Are there HIDDEN shared dependencies? (like two "different"
 *     servers that are secretly in the same rack, sharing the same
 *     power supply — or two regions that both use the same DNS provider)
 *
 *
 * ============================================================================
 * STRATEGIES TO REMOVE SPOFs
 * ============================================================================
 *
 * There's no one-size-fits-all fix. Different strategies fit different
 * components. Let's go through the main toolbox.
 *
 * -----------------------------------------------------------------------
 * STRATEGY 1: REDUNDANCY (just... have more than one)
 * -----------------------------------------------------------------------
 * The most obvious fix: don't have just ONE of something. But HOW you
 * add extras matters a lot:
 *
 *   Type            | What it means                          | Failover speed | Cost
 *   -----------------|------------------------------------------|-----------------|--------
 *   Active-Active    | ALL copies handle traffic at once         | Nearly instant  | Higher
 *   Active-Passive   | One works, one sits ready as backup       | Seconds-minutes | Medium
 *   N+1              | You need N, but you deploy N+1 (1 spare)  | Nearly instant  | Medium
 *   N+2              | Extra buffer, even during maintenance     | Nearly instant  | Higher
 *
 *   ACTIVE-ACTIVE PICTURE:
 *
 *      [User] -> [Load Balancer] -> [App Server 1]  (working)
 *                                 -> [App Server 2]  (working)
 *                                 -> [App Server 3]  (FAILS)
 *
 *      Servers 1 and 2 just keep going. Users barely notice, AS LONG AS
 *      the remaining servers have enough spare capacity to absorb the load.
 *
 *   ACTIVE-PASSIVE PICTURE:
 *
 *      NORMAL:  [User] -> [Primary DB] --(copies data to)--> [Standby DB]
 *      AFTER PRIMARY DIES: [User] -> [Standby DB] (now promoted to Primary)
 *
 *      Cheaper because the standby just sits there quietly copying data
 *      until it's needed — but the switch-over takes a little time.
 *
 * -----------------------------------------------------------------------
 * STRATEGY 2: LOAD BALANCING (with real health checks)
 * -----------------------------------------------------------------------
 * A load balancer isn't just "spreads traffic around" — a GOOD one also
 * constantly checks "is this server actually healthy?" and automatically
 * stops sending traffic to broken ones.
 *
 * ARCHITECTURE PICTURE:
 *
 *   [User] -> [DNS] -> [Load Balancer] -> [Server 1: healthy] ✓
 *                                       -> [Server 2: healthy] ✓
 *                                       -> [Server 3: FAILED] -> removed!
 *
 * Things to actually configure (not just "turn on"):
 *   - Health checks: don't just check "can I connect?" — check "can this
 *     server actually DO its job?" (a real endpoint test, not just a ping)
 *   - Automatic removal: unhealthy servers get pulled out within seconds
 *   - Graceful draining: let EXISTING connections finish before yanking
 *     a server out, instead of cutting people off mid-request
 *   - Session affinity (if needed): keep sending the same user to the
 *     same server, when that matters for your app
 *
 *   But wait... what if the LOAD BALANCER itself breaks? That's the next
 *   problem to solve.
 *
 * -----------------------------------------------------------------------
 * STRATEGY 3: REDUNDANT LOAD BALANCERS
 * -----------------------------------------------------------------------
 * A single load balancer just MOVED the SPOF, it didn't remove it. Here
 * are 3 common fixes:
 *
 *   A) VIRTUAL IP (VIP) with a tool like "Keepalived"
 *      Two load balancers SHARE one virtual address. The primary owns it
 *      and handles traffic; if it dies, the standby grabs the address
 *      and takes over — usually within a few seconds.
 *
 *      PICTURE:
 *        [LB 1 - Primary, owns the shared address] <--watches-- [LB 2 - Standby]
 *        (if LB 1 dies, LB 2 instantly claims the address and takes over)
 *
 *   B) DNS ROUND ROBIN + health checks
 *      DNS just hands out multiple IP addresses, and a health-checker
 *      removes broken ones from the list. Simpler to set up, but slower,
 *      because DNS changes take time to spread across the internet.
 *
 *   C) MANAGED CLOUD LOAD BALANCERS (AWS ALB, Google Cloud LB, Azure LB)
 *      The cloud provider handles the redundancy FOR you. Easiest option
 *      for most teams — but remember, you're now depending on THAT
 *      provider, region, and configuration being correct.
 *
 * -----------------------------------------------------------------------
 * STRATEGY 4: DATABASE REPLICATION (the hard one — because data is stateful)
 * -----------------------------------------------------------------------
 * Databases are trickier than app servers. An app server can just be
 * thrown away and rebuilt — it doesn't remember anything important. A
 * database HOLDS your actual data, so failover has to be careful not to
 * lose or corrupt anything.
 *
 *   PRIMARY-REPLICA REPLICATION (most common pattern):
 *   All writes go to ONE primary. The primary copies ("replicates") that
 *   data out to several replicas. Replicas handle read traffic, and if
 *   the primary dies, one replica gets "promoted" to become the new primary.
 *
 *   PICTURE:
 *
 *        [Writes] -> [Primary] --copies to--> [Replica 1] (handles reads)
 *                                           -> [Replica 2] (handles reads)
 *                                           -> [Replica 3] (handles reads)
 *
 *   Good for scaling reads and gives you a failover option — but until
 *   the failover finishes, the primary is still your ONLY place writes
 *   can go. That's a temporary but real risk window.
 *
 *   MULTI-PRIMARY REPLICATION (for write-heavy or global apps):
 *   Multiple primaries can ALL accept writes and sync with each other.
 *
 *   PICTURE:
 *
 *        [Primary 1] <--sync--> [Primary 2]
 *          |                        |
 *        [Replica 1]             [Replica 2]
 *
 *   Removes the "one writable node" risk, but now you have a NEW problem:
 *   what if the SAME piece of data gets changed on both primaries at the
 *   same time? You need "conflict resolution" logic to sort that out —
 *   which adds real complexity.
 *
 *   TRADE-OFF TABLE:
 *
 *     Approach                | Consistency        | Write availability      | Complexity
 *     --------------------------|----------------------|---------------------------|-------------
 *     Single Primary + Replicas | Strong               | Writes pause on failover  | Low
 *     Multi-Primary              | Usually "eventual"   | High                      | High
 *     Consensus (Raft/Paxos)     | Strong               | Needs majority of nodes   | Medium
 *
 *   For MOST apps: single primary with automatic failover is the sweet
 *   spot. Only reach for multi-primary if you genuinely need writes
 *   happening in multiple locations AND you have a real plan for
 *   resolving conflicts.
 *
 * -----------------------------------------------------------------------
 * STRATEGY 5: MULTI-ZONE DEPLOYMENT
 * -----------------------------------------------------------------------
 * Cloud providers split their infrastructure into "regions," and each
 * region has multiple "availability zones" — basically separate physical
 * data centers (different power, different cooling, different network)
 * that are still close enough to talk to each other quickly.
 *
 * PICTURE:
 *
 *   REGION: US-East
 *     Zone A: [Server] [DB Primary]  [Cache]
 *     Zone B: [Server] [DB Replica]  [Cache]
 *     Zone C: [Server] [DB Replica]  [Cache]
 *
 * If Zone A loses power entirely, Zones B and C can KEEP running — as
 * long as they have enough spare capacity and the failover is actually
 * configured correctly. This is a common, sensible baseline for most
 * production apps — it protects against a LOT of failures without the
 * cost/complexity of going full multi-region.
 *
 * -----------------------------------------------------------------------
 * STRATEGY 6: MULTI-REGION DEPLOYMENT
 * -----------------------------------------------------------------------
 * This means running your FULL stack in multiple, geographically
 * separate regions (like US-East AND Europe).
 *
 * PICTURE:
 *
 *   [Global DNS] --routes to--> [US-East: full app+db+cache stack]
 *                --routes to--> [EU-West: full app+db+cache stack]
 *                     (with data syncing between them)
 *
 * This protects against a WHOLE region going down (natural disasters,
 * big provider outages) and can even improve speed for users around the
 * world. But it's expensive and complicated:
 *   - Data consistency gets hard (strong consistency = slower writes;
 *     eventual consistency = possible conflicts)
 *   - Cross-region communication is genuinely slow (100ms+ extra delay)
 *   - You're paying for FULL infrastructure in each region
 *   - Deployments, monitoring, debugging — all harder
 *
 * Only go multi-region when the BUSINESS actually needs that level of
 * protection. For a lot of apps, multi-zone (above) is already enough.
 *
 * -----------------------------------------------------------------------
 * STRATEGY 7: ELIMINATE HUMAN SPOFs (yes, PEOPLE can be a SPOF too!)
 * -----------------------------------------------------------------------
 * All this technical redundancy means nothing if only ONE person on your
 * team actually knows how to operate the system.
 *
 * WARNING SIGNS:
 *   - "Only John knows how to deploy to production"
 *   - "Sarah wrote that service, only she really understands it"
 *   - "We can't release anything while Mike is on vacation"
 *
 * FIXES:
 *
 *   Problem                  | Fix
 *   ---------------------------|----------------------------------------
 *   Single expert              | Cross-training, pair programming, rotation
 *   Undocumented systems       | Runbooks, architecture docs, decision logs
 *   Manual processes           | Automate with CI/CD pipelines
 *   Single point of access     | Shared credential vault, role-based access
 *
 *   The goal isn't "make people replaceable like robots" — it's making
 *   sure KNOWLEDGE is spread out, so no single person's vacation or sick
 *   day can freeze the whole team.
 *
 *
 * ============================================================================
 * LAYER-BY-LAYER: HOW TO FIX SPOFs AT EVERY LEVEL
 * ============================================================================
 *
 * ----- DNS LAYER -----
 * PROBLEM: If your DNS provider goes down, NOTHING else matters — nobody
 * can even find your app's address, no matter how redundant everything
 * else is.
 *
 * FIX: Use two DNS providers (like Route 53 AND Cloudflare) with health
 * checks to automatically fail over. Keep the "TTL" (how long DNS
 * answers are cached) low enough for fast failover, but not SO low that
 * you flood the system with constant lookups.
 *
 * PICTURE:
 *
 *   [example.com] -> [Primary DNS: Route 53] --healthy?--> yes: keep using
 *                                            --healthy?--> no: switch to
 *                  -> [Secondary DNS: Cloudflare]
 *
 * WATCH OUT: this only actually helps if both providers' records are
 * kept perfectly in sync, and the domain registrar itself isn't a hidden
 * SPOF underneath all of it.
 *
 * ----- CDN LAYER -----
 * PROBLEM: One CDN provider (or one origin server behind it) can become
 * a SPOF for all your images, scripts, and cached content.
 *
 * FIX: Split traffic across multiple CDN providers, and configure
 * multiple origin servers as backups.
 *
 * PICTURE:
 *
 *   [Users] -> [Primary CDN: 80% traffic]  -> [Origin Servers]
 *           -> [Secondary CDN: 20% traffic] -> [Origin Servers]
 *
 *   If the primary CDN fails, traffic can shift over — and running a
 *   small % through the secondary all the time means you KNOW it
 *   actually works, instead of finding out during an actual outage.
 *
 * ----- APPLICATION LAYER -----
 * PROBLEM: One app server = any crash, deploy, or resource spike takes
 * the WHOLE service down.
 *
 * FIX: Run multiple server instances behind a load balancer, use
 * auto-scaling to add more when demand spikes, and (ideally) use
 * container orchestration (like Kubernetes) to keep multiple copies
 * ("replicas") running automatically.
 *
 * IMPORTANT DESIGN TIP: make your app "stateless" — meaning it doesn't
 * store important data locally on ITSELF (put session data somewhere
 * shared instead). That way, ANY server instance can handle ANY request,
 * and losing one instance is a non-event.
 *
 * ----- CACHE LAYER -----
 * PROBLEM: A single cache server creates TWO risks: (1) it can just
 * fail, and (2) when it fails, ALL the traffic it was absorbing suddenly
 * slams into your database, which might not survive that.
 *
 * FIX: Run a distributed cache cluster (like Redis Cluster or
 * Memcached), with replicas for each shard, and design your app to
 * degrade GRACEFULLY if the cache is briefly unavailable (rather than
 * just falling over).
 *
 * PICTURE:
 *
 *   Redis Cluster:
 *     [Primary 1] -> [Replica 1]
 *     [Primary 2] -> [Replica 2]
 *     [Primary 3] -> [Replica 3]
 *
 *   Data is split ("sharded") across the primaries. If one primary
 *   fails, its replica can take over — but keys owned by that shard
 *   might have a brief gap until failover finishes.
 *
 * ----- DATABASE LAYER -----
 * PROBLEM: Usually the SCARIEST SPOF, because it holds your actual data
 * — the "source of truth." Other layers can be rebuilt from scratch;
 * database problems can mean permanent data loss.
 *
 * FIXES:
 *   1. Read Replicas — scale reads AND give you failover candidates
 *      (see the Strategy 4 picture above)
 *   2. Automatic Failover — tools like Orchestrator (MySQL), Patroni
 *      (PostgreSQL), or managed services (like AWS RDS) can promote a
 *      replica automatically. Still double-check: is the replica's data
 *      actually up to date? Could TWO nodes both think they're primary
 *      ("split-brain")? Do clients reconnect properly afterward?
 *   3. Distributed Databases — built for high availability from day one:
 *
 *        Database    | Approach              | Consistency
 *        -------------|-------------------------|---------------
 *        CockroachDB  | Raft consensus          | Strong
 *        Cassandra    | Tunable consistency     | Configurable
 *        MongoDB      | Replica sets            | Configurable
 *        Spanner      | TrueTime + Paxos        | Strong
 *
 * ----- MESSAGE QUEUE LAYER -----
 * PROBLEM: One message broker dying can block BOTH the producers
 * (things sending messages) and the consumers (things reading them).
 *
 * FIX: Run a clustered broker setup (Kafka or RabbitMQ cluster), with
 * topics/queues REPLICATED across multiple brokers, and clients
 * configured to know about multiple broker addresses so they can
 * reconnect elsewhere automatically.
 *
 * PICTURE:
 *
 *   Kafka Cluster: [Broker 1] [Broker 2] [Broker 3]
 *   Topic "orders" -> replicated across all 3 (replication factor: 3)
 *
 *   If Broker 1 dies, messages are STILL available on Brokers 2 and 3,
 *   as long as replication and "quorum" settings are configured right.
 *
 * ----- STORAGE LAYER -----
 * PROBLEM: One disk or storage volume is a SPOF for BOTH availability
 * (can you access the data right now) AND durability (does the data
 * still exist at all).
 *
 * FIX: Use RAID for local disks, or reach for distributed cloud storage
 * (S3, Google Cloud Storage, Azure Blob) which already spreads data
 * across multiple zones. Add cross-region replication for extra
 * disaster-recovery protection.
 *
 * PICTURE:
 *
 *   [Upload] -> [Primary Region storage] --async replication--> [Secondary Region storage]
 *
 *
 * ============================================================================
 * TESTING FOR SPOFs (don't just ASSUME your fix works!)
 * ============================================================================
 *
 * Having redundancy drawn on a diagram means nothing if you've never
 * actually tested that the failover works. The WORST time to discover a
 * broken failover is during a real outage at 3 AM.
 *
 * ----- CHAOS ENGINEERING -----
 * This means deliberately BREAKING things on purpose, in a controlled
 * way, to see if your system actually survives like you think it will.
 *
 *   Test                     | What it proves
 *   ---------------------------|--------------------------------------
 *   Kill random instances      | Does auto-scaling/load balancing work?
 *   Kill the database primary  | Does automatic failover actually work?
 *   Block network to a zone    | Does multi-zone resilience hold up?
 *   Overload CPU/memory        | Does the system degrade gracefully?
 *   Drop DNS responses         | Does DNS redundancy kick in?
 *
 * Start in a safe test environment, then GRADUALLY move to testing in
 * production during low-traffic times, with a clear "abort button" ready
 * if things go sideways. The goal is real confidence, not chaos for fun.
 *
 * ----- GAME DAYS -----
 * A scheduled team exercise where you SIMULATE a failure together:
 *   1. Pick a failure scenario (e.g., "database primary dies")
 *   2. Predict what SHOULD happen, based on your architecture
 *   3. Actually trigger the failure (safely)
 *   4. Watch what REALLY happens and compare it to your prediction
 *   5. Write down the gaps and go fix them
 *
 *   This tests more than just the SYSTEM — it also tests the PEOPLE.
 *   Does on-call actually know what to do? Are the runbooks correct?
 *   Can the team communicate well under pressure?
 *
 * ----- AUTOMATED TESTING -----
 * Build failover tests right into your CI/CD pipeline so failover
 * regressions get caught automatically, before they ever hit production.
 * These take longer to run, so they usually run nightly rather than on
 * every single commit — but catching a broken failover ahead of time is
 * always worth the extra wait.
 *
 *
 * ============================================================================
 * COST VS. RESILIENCE (you can't fix EVERYTHING — so prioritize)
 * ============================================================================
 *
 * Removing every single SPOF everywhere would be incredibly expensive,
 * and honestly, not every SPOF deserves the same level of concern. A
 * SPOF in your PAYMENT flow matters a LOT more than a SPOF in your
 * internal admin dashboard that only 3 employees use.
 *
 * RISK ASSESSMENT TABLE (example):
 *
 *   Component     | Impact if it fails | How likely to fail | Cost to fix | Decision
 *   ----------------|-----------------------|-----------------------|----------------|------------
 *   Database        | Critical              | Medium                | High           | Fix it (replicate)
 *   App server       | High                  | Medium                | Low            | Fix it (redundant)
 *   Dev environment  | Low                   | Medium                | Medium         | Leave it single, it's fine
 *   Analytics DB      | Medium                | Low                   | High           | Maybe later
 *
 * The rough idea: weigh "how bad would it be if this breaks" against
 * "how much would it cost to make it redundant." Sometimes the math
 * clearly says "fix it now." Other times it tells you a simple recovery
 * PLAN (rather than full redundancy) is good enough.
 *
 * START SIMPLE, GROW OVER TIME — you don't need to fix everything on day one:
 *
 *   PHASE 1 (the critical path — do this first):
 *     - Database replication with automatic failover
 *     - Multiple app servers behind a load balancer
 *     - A managed load balancer (let the cloud provider handle its redundancy)
 *
 *   PHASE 2 (enhanced resilience — once you're more established):
 *     - Multi-zone deployment within your main region
 *     - Cache clustering
 *     - Automated alerting + written runbooks
 *
 *   PHASE 3 (maximum availability — for mature, large-scale systems):
 *     - Multi-region deployment
 *     - Multi-CDN strategy
 *     - Regular chaos engineering exercises
 *
 *   Each phase is a real jump in cost and complexity — only move forward
 *   when the BUSINESS actually needs it, not just because it sounds cool.
 *
 *
 * ============================================================================
 * BIG TAKEAWAYS (remember these if nothing else)
 * ============================================================================
 *
 *   1. A SPOF isn't "having only one of something" — it's having only
 *      one of something AND no automatic backup plan for when it breaks.
 *
 *   2. Hunt for SPOFs by asking "what if THIS fails?" for every single
 *      piece of your system — and then follow the chain: what does IT
 *      break next?
 *
 *   3. Redundancy at one layer doesn't mean redundancy everywhere. Check
 *      every layer separately (network, app, cache, database, storage,
 *      DNS, region, and even PEOPLE).
 *
 *   4. Different components need different fixes: load balancers need
 *      health checks, databases need careful replication, and humans
 *      need documentation and cross-training.
 *
 *   5. Don't just assume your redundancy works — TEST it, on purpose,
 *      through chaos engineering and game days, before a real outage
 *      forces you to find out the hard way.
 *
 *   6. You can't (and shouldn't) fix every SPOF at once. Prioritize by
 *      actual business impact, and grow your resilience in phases as
 *      the system and the company matures.
 *
 *   The real goal isn't "never fail" — that's impossible. The real goal
 *   is: no SINGLE failure should ever be able to take down the whole
 *   system. Failures will happen. Make sure they stay small.
 *
 * ============================================================================
 */